In [1]:
import pandas as pd
print("notebook works")

notebook works


In [2]:
day = pd.read_csv("data/scans_2026-05-04.csv", dtype=str) 
day.info() 
day["scan_type"].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 1285 entries, 0 to 1284
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   scan_id       1285 non-null   str  
 1   parcel_id     1285 non-null   str  
 2   customer_id   1285 non-null   str  
 3   scanned_at    1285 non-null   str  
 4   hub_id        1285 non-null   str  
 5   courier_id    1274 non-null   str  
 6   scan_type     1285 non-null   str  
 7   weight_kg     1276 non-null   str  
 8   service_code  1285 non-null   str  
dtypes: str(9)
memory usage: 160.9 KB


scan_type
DEPART              220
ARRIVE              217
FAILED              212
OUT_FOR_DELIVERY    211
PICKUP              203
DELIVERED           195
delivered             8
failed                6
pickup                4
arrive                3
out_for_delivery      3
depart                3
Name: count, dtype: int64

In [7]:
import sys, requests, time
import pandas as pd
from sqlalchemy import create_engine

# The API lives in the api/ folder — start it in-process, exactly like Lecture 7
sys.path.insert(0, "api")
from waseet_api import start_api

BASE_URL = start_api()
print("API on", BASE_URL)

# health check first — no key needed
health = requests.get(BASE_URL + "/health", timeout=5)
print(health.status_code, health.json())

API on http://127.0.0.1:55314
200 {'status': 'up', 'customers': 200}


In [8]:
from waseet_api import reset_rate_limit
reset_rate_limit()

headers = {"X-API-Key": "waseet-demo-key"}
first = requests.get(BASE_URL + "/customers", params={"page": 1}, headers=headers, timeout=10)
print("status:", first.status_code)
payload = first.json()
print("envelope keys:", list(payload.keys()))
print("records on page 1:", len(payload["customers"]))
print("total_pages:", payload["total_pages"], "| has_more:", payload["has_more"])

status: 200
envelope keys: ['page', 'page_size', 'total_pages', 'total_records', 'has_more', 'customers']
records on page 1: 25
total_pages: 8 | has_more: True


In [9]:
def get_with_retry(url, headers, params, attempts=4):
    """One GET, with a timeout, and a doubling wait on 429 and 5xx.
    Retry the transient; give up on the permanent (401/404 will never work)."""
    wait = 1
    for attempt in range(attempts):
        reply = requests.get(url, headers=headers, params=params, timeout=10)
        if reply.status_code == 429 or reply.status_code >= 500:
            print("  transient", reply.status_code, "on attempt", attempt + 1, "- waiting", wait, "s")
            time.sleep(wait)
            wait = wait * 2
            continue
        return reply           # 200, or a permanent error like 401 — return it, don't retry
    return reply

In [10]:
def fetch_all_customers(base_url):
    headers = {"X-API-Key": "waseet-demo-key"}
    page = 1
    records = []
    while True:
        reply = get_with_retry(base_url + "/customers", headers, {"page": page})
        reply.raise_for_status()          # turn a permanent error into a stop
        body = reply.json()
        records.extend(body["customers"]) # records live under "customers"
        if not body["has_more"]:
            break
        page = page + 1
    return records

reset_rate_limit()                        # so the 429s happen, to prove retry works
records = fetch_all_customers(BASE_URL)
print("fetched", len(records), "customers")

  transient 429 on attempt 1 - waiting 1 s
  transient 429 on attempt 1 - waiting 1 s
fetched 200 customers


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 50060)
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\socketserver.py", line 766, in __init__
    self.handle()
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\http\server.py", line 436, in handle
    self.handle_one_request()
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\http\server.py", line 

In [11]:
from sqlalchemy import create_engine

DB_URL = "postgresql+psycopg2://de:de@localhost:5442/waseet"
engine = create_engine(DB_URL)

def load_customers(records):
    """Land them in the customers table. Idempotent: delete then insert,
    so running twice does not double the dimension and does not fail on the PK."""
    df = pd.DataFrame(records)
    with engine.begin() as conn:
        conn.exec_driver_sql("DELETE FROM customers")   # clear first -> repeatable
        df.to_sql("customers", conn, if_exists="append", index=False)
    return len(df)

n = load_customers(records)
print("loaded", n, "customers")

loaded 200 customers


In [12]:
import pandas as pd
from sqlalchemy import create_engine, text

DATA_DIR = "data"
DB_URL = "postgresql+psycopg2://de:de@localhost:5442/waseet"
engine = create_engine(DB_URL)

# The contract: the 9 columns a scan file must have. The 19th renames weight_kg.
REQUIRED = ["scan_id", "parcel_id", "customer_id", "scanned_at",
            "hub_id", "courier_id", "scan_type", "weight_kg", "service_code"]

def extract(scan_date):
    """Read one day's file as strings — dtype=str so pandas doesn't reinterpret
    things like '12,5' before we've looked at them."""
    path = DATA_DIR + "/scans_" + scan_date + ".csv"
    df = pd.read_csv(path, dtype=str)
    print("extract:", len(df), "rows from", path)
    return df

def check_contract(df):
    """Fail fast if the columns aren't the ones we agreed, or the file is empty.
    Cheap, first, before the transform touches anything."""
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError("missing columns: " + str(missing))
    if len(df) == 0:
        raise ValueError("file is empty")
    print("contract: ok,", len(df.columns), "columns")

# Test on a normal day
raw = extract("2026-05-04")
check_contract(raw)

extract: 1285 rows from data/scans_2026-05-04.csv
contract: ok, 9 columns


In [13]:
# The 19th: renamed column — contract must fail it
try:
    bad = extract("2026-05-19")
    check_contract(bad)
except ValueError as e:
    print("19th correctly rejected ->", e)

# The 15th: empty (Eid) — contract flags it empty (the DAG will skip it before this)
try:
    empty = extract("2026-05-15")
    check_contract(empty)
except ValueError as e:
    print("15th correctly flagged ->", e)

extract: 1066 rows from data/scans_2026-05-19.csv
19th correctly rejected -> missing columns: ['weight_kg']
extract: 0 rows from data/scans_2026-05-15.csv
15th correctly flagged -> file is empty


In [14]:
import os
os.makedirs("quarantine", exist_ok=True)

VALID_TYPES = ["PICKUP", "ARRIVE", "DEPART", "OUT_FOR_DELIVERY", "DELIVERED", "FAILED"]

def parse_timestamps(series):
    """Two formats arrive. Return one datetime column.
    ~3% use DD/MM/YYYY HH:MM instead of ISO. coerce gives NaT (not an error),
    which is what lets fillna merge the two."""
    iso   = pd.to_datetime(series, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    other = pd.to_datetime(series, format="%d/%m/%Y %H:%M",   errors="coerce")
    return iso.fillna(other)

def transform(raw, scan_date, hubs, couriers, customers, services):
    """Return (good, rejects). Repairs are fixed and kept; rejects are
    quarantined with a reason written on each row."""
    df = raw.copy()

    # --- REPAIRS (fix and keep) ---
    # 1. timestamps: two formats -> one datetime
    df["scanned_at"] = parse_timestamps(df["scanned_at"])
    # 2. scan_type mixed case -> uppercase
    df["scan_type"] = df["scan_type"].str.upper()
    # 3. weight comma-decimal -> dot, then numeric (blank/comma become NaN cleanly)
    df["weight_kg"] = pd.to_numeric(
        df["weight_kg"].str.replace(",", ".", regex=False), errors="coerce")
    # 4. numeric ids
    for col in ["scan_id", "parcel_id", "customer_id", "hub_id", "courier_id"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # dedup: the same event sent twice (and the doubled 13th) -> keep first
    before = len(df)
    df = df.drop_duplicates(subset=["scan_id"], keep="first")
    deduped = before - len(df)

    # --- REJECTS (quarantine with a reason) ---
    df["reject_reason"] = None
    df.loc[df["scanned_at"].isna(), "reject_reason"] = "unparseable timestamp"
    df.loc[~df["hub_id"].isin(hubs["hub_id"]), "reject_reason"] = "unknown hub"
    df.loc[(df["weight_kg"] > 100) | (df["weight_kg"] <= 0), "reject_reason"] = "impossible weight"
    df.loc[~df["scan_type"].isin(VALID_TYPES), "reject_reason"] = "unknown scan_type"
    df.loc[~df["customer_id"].isin(customers["customer_id"]), "reject_reason"] = "unknown customer"
    df.loc[~df["service_code"].isin(services["service_code"]), "reject_reason"] = "unknown service"
    # courier: blank is VALID (unassigned) -> leave as NaN, do NOT reject.
    # but a courier_id that is present and unknown IS a reject:
    bad_courier = df["courier_id"].notna() & ~df["courier_id"].isin(couriers["courier_id"])
    df.loc[bad_courier, "reject_reason"] = "unknown courier"

    good = df[df["reject_reason"].isna()].copy()
    rejects = df[df["reject_reason"].notna()].copy()

    if len(rejects) > 0:
        rejects.to_csv("quarantine/rejects_" + scan_date + ".csv", index=False)

    # derive scan_date (the column we index/partition on)
    good["scan_date"] = good["scanned_at"].dt.date

    print(f"transform: {len(raw)} read = {len(good)} good + {len(rejects)} rejected + {deduped} deduped")
    return good, rejects

In [15]:
hubs      = pd.read_csv("data/hubs.csv")
couriers  = pd.read_csv("data/couriers.csv")
services  = pd.read_csv("data/service_levels.csv")
customers = pd.read_sql_query("SELECT customer_id FROM customers", engine)

raw = extract("2026-05-04")
good, rejects = transform(raw, "2026-05-04", hubs, couriers, customers, services)

print()
print("reject reasons breakdown:")
print(rejects["reject_reason"].value_counts())

extract: 1285 rows from data/scans_2026-05-04.csv
transform: 1285 read = 1262 good + 20 rejected + 3 deduped

reject reasons breakdown:
reject_reason
impossible weight    12
unknown hub           8
Name: count, dtype: int64


In [16]:
def load(good, scan_date):
    """Write one day idempotently: delete the day first, then insert.
    Running twice for the same date leaves the warehouse exactly as running once.
    This also handles the file that was sent twice (13th) — dedup already
    collapsed it, and the delete-first makes a re-run safe."""
    cols = ["scan_id", "parcel_id", "customer_id", "scanned_at", "scan_date",
            "hub_id", "courier_id", "scan_type", "weight_kg", "service_code"]
    out = good[cols].copy()
    # courier_id is nullable — turn NaN into a proper NULL for Postgres
    out["courier_id"] = out["courier_id"].astype("Int64")
    with engine.begin() as conn:
        conn.execute(text("DELETE FROM parcel_scans WHERE scan_date = :d"), {"d": scan_date})
        out.to_sql("parcel_scans", conn, if_exists="append", index=False)
    print("load:", len(out), "rows for", scan_date)
    return len(out)

# Test it — load the 4th
n = load(good, "2026-05-04")

load: 1262 rows for 2026-05-04


In [17]:
# load the 4th again — should still end at 1262, not 2524
load(good, "2026-05-04")
count = pd.read_sql_query(
    "SELECT count(*) AS n FROM parcel_scans WHERE scan_date = '2026-05-04'", engine)
print("rows in warehouse for the 4th:", count["n"][0])

load: 1262 rows for 2026-05-04
rows in warehouse for the 4th: 1262


In [23]:
import pandas as pd
from sqlalchemy import create_engine
engine = create_engine("postgresql+psycopg2://de:de@localhost:5442/waseet")

VALID_TYPES = ["PICKUP", "ARRIVE", "DEPART", "OUT_FOR_DELIVERY", "DELIVERED", "FAILED"]

SUITE = [
    {"column": "scan_id",      "check": "not_null",  "dimension": "completeness", "where": "transform"},
    {"column": "scan_id",      "check": "unique",    "dimension": "uniqueness",   "where": "transform"},
    {"column": "scanned_at",   "check": "not_null",  "dimension": "completeness", "where": "transform"},
    {"column": "customer_id",  "check": "numeric",   "dimension": "validity",     "where": "transform"},
    {"column": "hub_id",       "check": "numeric",   "dimension": "validity",     "where": "transform"},
    {"column": "weight_kg",    "check": "in_range",  "min": 0, "max": 100,
                                                     "dimension": "validity",     "where": "transform"},
    {"column": "scan_type",    "check": "in_set",    "allowed": VALID_TYPES,
                                                     "dimension": "consistency",  "where": "transform"},
    {"column": "service_code", "check": "in_set",    "allowed": ["SDD","EXP","STD","ECO"],
                                                     "dimension": "validity",     "where": "transform"},
    {"column": "scan_type",    "check": "uppercase", "dimension": "consistency",  "where": "transform"},
]

def run_check(df, rule):
    col = rule["column"]; check = rule["check"]; s = df[col]
    if check == "not_null":
        bad = s.isna() | (s.astype(str).str.strip() == "")
    elif check == "unique":
        bad = s.duplicated(keep=False)
    elif check == "numeric":
        bad = pd.to_numeric(s, errors="coerce").isna() & s.notna()
    elif check == "in_range":
        n = pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")
        bad = (n < rule["min"]) | (n > rule["max"])
    elif check == "in_set":
        bad = ~s.str.upper().isin([a.upper() for a in rule["allowed"]]) & s.notna()
    elif check == "uppercase":
        bad = s.notna() & (s != s.str.upper())
    else:
        raise ValueError("unknown check: " + check)
    return int(bad.sum())

print("suite has", len(SUITE), "rules")

suite has 9 rules


In [24]:
def validate(df, suite):
    rows = []
    for rule in suite:
        failed = run_check(df, rule)
        rows.append({"column": rule["column"], "check": rule["check"],
                     "dimension": rule["dimension"], "failed_rows": failed})
    return pd.DataFrame(rows)

day = pd.read_csv("data/scans_2026-05-04.csv", dtype=str)
report = validate(day, SUITE)
print(report.to_string(index=False))


      column     check    dimension  failed_rows
     scan_id  not_null completeness            0
     scan_id    unique   uniqueness            6
  scanned_at  not_null completeness            0
 customer_id   numeric     validity            0
      hub_id   numeric     validity            0
   weight_kg  in_range     validity           12
   scan_type    in_set  consistency            0
service_code    in_set     validity            0
   scan_type uppercase  consistency           27


In [25]:
import json, os

def write_baseline(df, baseline_path):
    """Save the column list of a known-good file as the baseline to compare against."""
    with open(baseline_path, "w") as f:
        json.dump(list(df.columns), f)

def check_schema(df, baseline_path):
    """Return (missing, new). Missing columns and new columns are different
    severities, so return them separately, not as one verdict."""
    with open(baseline_path) as f:
        baseline = json.load(f)
    missing = [c for c in baseline if c not in df.columns]
    new = [c for c in df.columns if c not in baseline]
    return missing, new

# Write the baseline from a known-good day (the 4th)
good_day = pd.read_csv("data/scans_2026-05-04.csv", dtype=str)
write_baseline(good_day, "schema_baseline.json")
print("baseline saved:", list(good_day.columns))

# Now test drift on the 19th (renamed column) — must catch it
day19 = pd.read_csv("data/scans_2026-05-19.csv", dtype=str)
missing, new = check_schema(day19, "schema_baseline.json")
print("19th -> missing:", missing, "| new:", new)

baseline saved: ['scan_id', 'parcel_id', 'customer_id', 'scanned_at', 'hub_id', 'courier_id', 'scan_type', 'weight_kg', 'service_code']
19th -> missing: ['weight_kg'] | new: ['weight']


In [26]:
import glob

def reconcile(scan_date, engine):
    """Compare file rows vs warehouse rows for one date. A gap is not a failure;
    an unexplained gap is. Return the pieces a human needs to tell them apart."""
    path = "data/scans_" + scan_date + ".csv"
    try:
        raw = pd.read_csv(path, dtype=str)
    except FileNotFoundError:
        return {"scan_date": scan_date, "file_rows": None, "wh_rows": None,
                "diff": None, "note": "file never arrived"}
    file_rows = len(raw)
    wh = pd.read_sql_query(
        "SELECT count(*) AS n FROM parcel_scans WHERE scan_date = %(d)s",
        engine, params={"d": scan_date})
    wh_rows = int(wh["n"][0])
    # how many of the file's rows we EXPECT not to reach the warehouse:
    dupes = raw["scan_id"].duplicated().sum() if "scan_id" in raw.columns else 0
    diff = file_rows - wh_rows
    return {"scan_date": scan_date, "file_rows": file_rows, "wh_rows": wh_rows,
            "diff": diff, "file_dupes": int(dupes)}

# Run reconciliation for all 21 days
rows = []
for d in range(1, 22):
    day = "2026-05-%02d" % d
    rows.append(reconcile(day, engine))

recon = pd.DataFrame(rows)
print(recon.to_string(index=False))

 scan_date  file_rows  wh_rows   diff  file_dupes               note
2026-05-01     1230.0   1204.0   26.0         5.0                NaN
2026-05-02     1164.0   1140.0   24.0         8.0                NaN
2026-05-03     1156.0   1139.0   17.0         4.0                NaN
2026-05-04     1285.0   1262.0   23.0         3.0                NaN
2026-05-05     1212.0   1187.0   25.0         8.0                NaN
2026-05-06     1291.0   1263.0   28.0         7.0                NaN
2026-05-07     1187.0   1166.0   21.0         8.0                NaN
2026-05-08     1205.0   1181.0   24.0         5.0                NaN
2026-05-09     1307.0   1284.0   23.0         9.0                NaN
2026-05-10        NaN      NaN    NaN         NaN file never arrived
2026-05-11     1242.0   1217.0   25.0         5.0                NaN
2026-05-12     1103.0   1084.0   19.0         3.0                NaN
2026-05-13     2546.0   1249.0 1297.0      1282.0                NaN
2026-05-14     1063.0   1044.0   1